# Symbolic Execution with Guppy

This notebook demonstrates the symbolic stabilizer simulation workflow:

1. **Write quantum circuits in Guppy** - Python-embedded quantum DSL
2. **Compile to HUGR** - Hierarchical Unified Graph Representation
3. **Execute symbolically** - Track measurement dependencies without collapsing
4. **Sample efficiently** - Generate millions of shots in milliseconds!

## Why Symbolic Execution?

Traditional simulators collapse measurements to concrete outcomes. This means:
- Each shot requires a full simulation
- 1 million shots = 1 million simulations

Symbolic execution instead tracks *dependencies* between measurements:
- Execute once, capturing which measurements depend on which
- Sampling becomes XOR operations on random bits
- 1 million shots ≈ milliseconds!

In [ ]:
from guppylang import guppy
from guppylang.std.quantum import cx, h, measure, qubit
from pecos.experimental import execute_hugr_symbolic

## Example 1: Bell State

The simplest entangled state - two qubits that always measure the same.

In [ ]:
@guppy
def bell_state() -> tuple[bool, bool]:
    """Create and measure a Bell state: (|00⟩ + |11⟩)/√2."""
    q0 = qubit()
    q1 = qubit()
    h(q0)        # Put q0 in superposition
    cx(q0, q1)   # Entangle q0 and q1
    return (measure(q0), measure(q1))

# Compile to HUGR
package = bell_state.compile()
hugr_bytes = package.to_bytes()
print(f"HUGR size: {len(hugr_bytes)} bytes")

In [ ]:
# Execute symbolically
result = execute_hugr_symbolic(hugr_bytes)

print(f"Symbolic result: {result}")
print("\nMeasurement structure:")
print(f"  Total measurements: {result.num_measurements}")
print(f"  Deterministic: {result.num_deterministic}")
print(f"  Non-deterministic (random): {result.num_nondeterministic}")

The output `[m0=?, m1=m0]` tells us:
- `m0=?` - First measurement is random (50/50)
- `m1=m0` - Second measurement always equals the first

This is the Bell state correlation!

In [ ]:
# Sample 1 million shots - this is extremely fast!
import time

num_shots = 1_000_000

start = time.perf_counter()
counts = result.sample_counts(num_shots)
elapsed = time.perf_counter() - start

print(f"Generated {num_shots:,} shots in {elapsed*1000:.2f} ms")
print("\nOutcome counts:")
for outcome, count in sorted(counts.items()):
    bits = "".join(str(b) for b in outcome)
    print(f"  |{bits}⟩: {count:,} ({100*count/num_shots:.1f}%)")

## Example 2: GHZ State

A 3-qubit entangled state: (|000⟩ + |111⟩)/√2

In [ ]:
@guppy
def ghz_state() -> tuple[bool, bool, bool]:
    """Create and measure a 3-qubit GHZ state: (|000⟩ + |111⟩)/√2."""
    q0 = qubit()
    q1 = qubit()
    q2 = qubit()
    h(q0)
    cx(q0, q1)
    cx(q1, q2)
    return (measure(q0), measure(q1), measure(q2))

# Compile and execute symbolically
result = execute_hugr_symbolic(ghz_state.compile().to_bytes())

print(f"Symbolic result: {result}")
print("\nInterpretation:")
print("  m0=? : First qubit is random")
print("  m1=m0: Second qubit equals first")
print("  m2=m0: Third qubit equals first (via transitivity)")

In [ ]:
# Sample and verify
counts = result.sample_counts(100_000)

print("GHZ state outcomes (should only see 000 and 111):")
for outcome, count in sorted(counts.items()):
    bits = "".join(str(b) for b in outcome)
    print(f"  |{bits}⟩: {count:,}")

## Example 3: Repetition Code Syndrome Extraction

A more realistic example: 3-qubit repetition code with syndrome measurements.

```
Data qubits: q0, q1, q2 (encode logical qubit)
Ancillas: q3 (Z0Z1 check), q4 (Z1Z2 check)

Logical |+_L⟩ = (|000⟩ + |111⟩)/√2
```

In [ ]:
@guppy
def repetition_code_logical_plus() -> tuple[bool, bool, bool, bool, bool]:
    """3-qubit repetition code in logical |+_L⟩ state with syndrome extraction.

    Returns: (syndrome0, syndrome1, data0, data1, data2)
    """
    # Data qubits
    d0 = qubit()
    d1 = qubit()
    d2 = qubit()
    # Ancilla qubits for syndrome measurement
    a0 = qubit()
    a1 = qubit()

    # Encode logical |+_L⟩ = (|000⟩ + |111⟩)/√2
    h(d0)
    cx(d0, d1)
    cx(d0, d2)

    # Syndrome extraction: Z0Z1 parity check
    # Uses CX gates to copy parity to ancilla
    cx(d0, a0)
    cx(d1, a0)
    s0 = measure(a0)

    # Syndrome extraction: Z1Z2 parity check
    cx(d1, a1)
    cx(d2, a1)
    s1 = measure(a1)

    # Measure data qubits
    m0 = measure(d0)
    m1 = measure(d1)
    m2 = measure(d2)

    return (s0, s1, m0, m1, m2)

# Execute symbolically
result = execute_hugr_symbolic(repetition_code_logical_plus.compile().to_bytes())

print(f"Symbolic result: {result}")
print("\nMeasurement structure:")
print(f"  Deterministic: {result.num_deterministic} (syndromes + correlated data)")
print(f"  Random: {result.num_nondeterministic} (logical qubit state)")

In [ ]:
# Sample and analyze
counts = result.sample_counts(100_000)

print("Repetition code outcomes (s0, s1, d0, d1, d2):")
print("Expected: syndromes=00, data qubits all same\n")

for outcome, count in sorted(counts.items(), key=lambda x: -x[1]):
    s0, s1, d0, d1, d2 = outcome
    syndrome = f"{s0}{s1}"
    data = f"{d0}{d1}{d2}"
    print(f"  syndrome={syndrome}, data={data}: {count:,}")

## Example 4: Comparing Sampling Performance

Let's compare the symbolic sampler against traditional simulation.

In [ ]:
import time

# Use the GHZ state for benchmarking
result = execute_hugr_symbolic(ghz_state.compile().to_bytes())

shot_counts = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

print("Symbolic sampling performance:")
print(f"{'Shots':>12} | {'Time (ms)':>10} | {'Shots/sec':>12}")
print("-" * 42)

for n in shot_counts:
    start = time.perf_counter()
    _ = result.sample_counts(n)
    elapsed = time.perf_counter() - start
    rate = n / elapsed
    print(f"{n:>12,} | {elapsed*1000:>10.2f} | {rate:>12,.0f}")

## How It Works: Measurement Dependencies

The symbolic simulator represents each measurement as:
- **Random bit index** (if non-deterministic) 
- **XOR of previous measurements** (dependencies)
- **Flip flag** (constant offset)

For example, in a Bell state:
- `m0 = random_bit[0]`
- `m1 = m0` (XOR of measurement 0, no flip)

Sampling just generates random bits and computes XORs - no matrix operations!

In [ ]:
# Get individual samples to see the structure
result = execute_hugr_symbolic(bell_state.compile().to_bytes())

print("First 10 samples from Bell state:")
samples = result.sample(10)
for i, sample in enumerate(samples):
    m0, m1 = sample
    print(f"  Shot {i}: m0={int(m0)}, m1={int(m1)} (equal: {m0 == m1})")

## Limitations

Symbolic execution works for **Clifford circuits** only:
- Supported gates: H, S, X, Y, Z, CX, CY, CZ, SWAP
- NOT supported: T gates, arbitrary rotations, non-Clifford gates

This is because Clifford circuits have efficient stabilizer representations,
while non-Clifford gates require exponential resources to simulate exactly.

In [ ]:
# This would fail with a non-Clifford gate:
# from guppylang.std.quantum import t
#
# @guppy
# def non_clifford():
#     q = qubit()
#     t(q)  # T gate is non-Clifford!
#     return measure(q)
#
# execute_hugr_symbolic(non_clifford.compile().to_bytes())
# RuntimeError: Unsupported gate for stabilizer simulation: T

## Summary

The `pecos.experimental` module provides:

- `execute_hugr_symbolic(hugr_bytes)` - Execute HUGR symbolically
- `execute_dag_circuit_symbolic(circuit)` - Execute DagCircuit directly
- `SymbolicExecutionResult` - Result with sampling methods:
  - `.sample(n)` - Get n samples as list of bool lists
  - `.sample_counts(n)` - Get n samples as outcome→count dict
  - `.num_measurements` - Total measurement count
  - `.num_deterministic` - Fixed/computed measurements
  - `.num_nondeterministic` - Random measurements

This enables extremely fast sampling for Clifford circuits, making it ideal for:
- Error correction simulations
- Syndrome extraction analysis
- Statistical sampling of stabilizer circuits

---

# Part 2: Noisy Symbolic Execution

The symbolic execution framework can also model **depolarizing noise**! Instead of tracking just measurement dependencies, we also track fault events - random errors that flip measurements.

## How Noisy Symbolic Execution Works

Traditional noisy simulation:
- Apply random Pauli errors after gates probabilistically
- Each shot requires full simulation with error injection
- 1 million shots = 1 million noisy simulations

Noisy symbolic execution:
- Track all *possible* fault locations and their propagation
- Faults become "hidden random bits" with probabilities
- Sampling draws fault bits (Bernoulli) and computes XORs
- 1 million shots ≈ milliseconds!

Each measurement becomes: `result = base_deps XOR fault_deps XOR flip`
- `base_deps`: XOR of random measurement bits (as before)
- `fault_deps`: XOR of fault bits that affect this measurement
- `flip`: constant offset

In [ ]:
# Import noisy execution functions
from pecos.experimental import execute_hugr_symbolic_noisy

## Noise Parameters

The noisy symbolic execution supports a depolarizing noise model with these parameters:

- **p1**: Single-qubit gate error probability (e.g., H, S, X, Y, Z)
- **p2**: Two-qubit gate error probability (e.g., CX, CZ)
- **p_meas**: Measurement error probability (bit flip)
- **p_prep**: State preparation error probability

When a single-qubit gate has error probability `p1`, each of the 3 Paulis (X, Y, Z) is applied with probability `p1/3`.

When a two-qubit gate has error probability `p2`, each of the 15 non-identity two-qubit Paulis is applied with probability `p2/15`.

In [ ]:
# Execute the Bell state with noise
noisy_result = execute_hugr_symbolic_noisy(
    hugr_bytes,
    p1=0.01,      # 1% single-qubit gate error
    p2=0.01,      # 1% two-qubit gate error
    p_meas=0.001, # 0.1% measurement error
    p_prep=0.001,  # 0.1% preparation error
)

print(f"Noisy symbolic result: {noisy_result}")
print(f"\nFault events: {noisy_result.num_faults}")
print(f"Measurements: {noisy_result.num_measurements}")

The noisy result now tracks fault events. With noise, we can observe:
- **00 and 11**: Still correlated (no fault or even number of flips)
- **01 and 10**: Anti-correlated (odd number of faults flipping one measurement)

In [ ]:
# Sample with noise
noisy_counts = noisy_result.sample_counts(100_000)

print("Noisy Bell state outcomes:")
for outcome, count in sorted(noisy_counts.items()):
    bits = "".join(str(int(b)) for b in outcome)
    print(f"  |{bits}>: {count:,} ({100*count/100_000:.2f}%)")

# Compare with noiseless
noiseless_counts = result.sample_counts(100_000)
print("\nNoiseless Bell state outcomes:")
for outcome, count in sorted(noiseless_counts.items()):
    bits = "".join(str(int(b)) for b in outcome)
    print(f"  |{bits}>: {count:,} ({100*count/100_000:.2f}%)")

## Comparison: Noisy Symbolic vs Traditional Simulation

Let's verify that noisy symbolic execution produces statistically equivalent results to traditional Monte Carlo simulation with depolarizing noise.

We'll:
1. Create a circuit in QASM (for traditional sim) and Guppy (for symbolic)
2. Run both with the same noise parameters
3. Compare measurement statistics

In [ ]:
# Import traditional simulation components
from collections import Counter

from pecos import Qasm, depolarizing_noise, sim

# Define equivalent Bell state circuit in QASM
bell_qasm = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

# Noise parameter
p = 0.05  # 5% error rate for clear effect

# Number of shots for comparison
num_shots = 50_000

In [ ]:
# Traditional simulation with depolarizing noise
print("Running traditional Monte Carlo simulation...")
start = time.perf_counter()

noise = depolarizing_noise().with_uniform_probability(p)
traditional_results = sim(Qasm.from_string(bell_qasm)).noise(noise).run(num_shots).to_dict()

traditional_time = time.perf_counter() - start
print(f"Traditional simulation: {traditional_time:.2f}s for {num_shots:,} shots")

# Convert to counts format (outcomes as tuples)
traditional_counts = Counter()
for i in range(len(traditional_results["c"])):
    # QASM returns measurements as integers, convert to bit tuples
    val = traditional_results["c"][i]
    outcome = (val & 1, (val >> 1) & 1)  # (c[0], c[1])
    traditional_counts[outcome] += 1

print("\nTraditional simulation outcomes:")
for outcome, count in sorted(traditional_counts.items()):
    bits = "".join(str(b) for b in outcome)
    print(f"  |{bits}>: {count:,} ({100*count/num_shots:.2f}%)")

In [ ]:
# Symbolic noisy execution
print("Running noisy symbolic execution...")
start = time.perf_counter()

# Use same noise probability for both single and two-qubit gates
symbolic_noisy_result = execute_hugr_symbolic_noisy(
    hugr_bytes,
    p1=p,
    p2=p,
)

symbolic_time_build = time.perf_counter() - start

# Now sample
start = time.perf_counter()
symbolic_noisy_counts = symbolic_noisy_result.sample_counts(num_shots)
symbolic_time_sample = time.perf_counter() - start

print(f"Symbolic build time: {symbolic_time_build*1000:.2f}ms")
print(f"Symbolic sample time: {symbolic_time_sample*1000:.2f}ms for {num_shots:,} shots")

print("\nSymbolic noisy outcomes:")
for outcome, count in sorted(symbolic_noisy_counts.items()):
    bits = "".join(str(int(b)) for b in outcome)
    print(f"  |{bits}>: {count:,} ({100*count/num_shots:.2f}%)")

## Statistical Comparison

Let's compute statistics to verify the distributions are equivalent:
- **Marginal probabilities**: P(m0=1), P(m1=1)
- **Correlation**: P(m0 = m1)
- **Error rate**: P(m0 != m1) (should be non-zero with noise)

In [ ]:
from math import sqrt


def compute_stats(counts: dict, num_shots: int) -> dict[str, float]:
    """Compute statistics from outcome counts."""
    # Marginal probabilities
    p_m0_1 = sum(c for (m0, m1), c in counts.items() if m0) / num_shots
    p_m1_1 = sum(c for (m0, m1), c in counts.items() if m1) / num_shots

    # Correlation (same outcome)
    p_same = sum(c for (m0, m1), c in counts.items() if m0 == m1) / num_shots

    # Anti-correlation (different outcome = error in Bell state)
    p_diff = sum(c for (m0, m1), c in counts.items() if m0 != m1) / num_shots

    return {
        "P(m0=1)": p_m0_1,
        "P(m1=1)": p_m1_1,
        "P(same)": p_same,
        "P(diff)": p_diff,
    }

# Convert symbolic counts to comparable format
symbolic_counts_converted = {}
for outcome, count in symbolic_noisy_counts.items():
    key = (int(outcome[0]), int(outcome[1]))
    symbolic_counts_converted[key] = count

trad_stats = compute_stats(traditional_counts, num_shots)
symb_stats = compute_stats(symbolic_counts_converted, num_shots)

print("Statistical Comparison")
print("=" * 50)
print(f"{'Statistic':<15} {'Traditional':>15} {'Symbolic':>15}")
print("-" * 50)
for stat in trad_stats:
    t_val = trad_stats[stat]
    s_val = symb_stats[stat]
    print(f"{stat:<15} {t_val:>15.4f} {s_val:>15.4f}")
print("-" * 50)

# Chi-squared test for goodness of fit
print("\nDifferences (should be small due to sampling noise):")
for stat in trad_stats:
    diff = abs(trad_stats[stat] - symb_stats[stat])
    # Expected statistical error ~ 1/sqrt(N)
    expected_error = 1 / sqrt(num_shots)
    print(f"  {stat}: {diff:.4f} (expected error ~{expected_error:.4f})")

## Performance Comparison

Now let's compare the sampling speed. The symbolic approach should be orders of magnitude faster!

In [ ]:
# Performance comparison
print("Performance Comparison")
print("=" * 60)

# Already measured above, but let's add more data points
shot_counts_test = [10_000, 50_000, 100_000, 500_000, 1_000_000]

print(f"\n{'Shots':>12} | {'Traditional':>15} | {'Symbolic':>15} | {'Speedup':>10}")
print("-" * 60)

for n_shots in shot_counts_test:
    # Traditional (estimate based on scaling)
    # Note: We only measure one to save time, then extrapolate
    trad_time = (
        traditional_time if n_shots == 50_000
        else traditional_time * (n_shots / num_shots)
    )

    # Symbolic
    start = time.perf_counter()
    _ = symbolic_noisy_result.sample_counts(n_shots)
    symb_time = time.perf_counter() - start

    speedup = trad_time / symb_time if symb_time > 0 else float("inf")

    print(f"{n_shots:>12,} | {trad_time:>12.3f}s | {symb_time*1000:>12.2f}ms | {speedup:>9.0f}x")

## Advanced: Larger Circuits with Noise

Let's try a larger circuit - the repetition code - with noise to see how the symbolic approach handles more complex correlations.

In [ ]:
# Compile repetition code
rep_code_bytes = repetition_code_logical_plus.compile().to_bytes()

# Execute with noise
rep_code_noisy = execute_hugr_symbolic_noisy(
    rep_code_bytes,
    p1=0.01,  # 1% single-qubit error
    p2=0.01,  # 1% two-qubit error
)

print("Repetition code noisy result:")
print(f"  Measurements: {rep_code_noisy.num_measurements}")
print(f"  Fault events: {rep_code_noisy.num_faults}")
print(f"\nSymbolic structure: {rep_code_noisy}")

In [ ]:
# Sample and analyze error correction properties
rep_noisy_counts = rep_code_noisy.sample_counts(100_000)

# Analyze syndrome and data patterns
syndrome_counts = Counter()
logical_errors = 0
total = 0

for outcome, count in rep_noisy_counts.items():
    s0, s1, d0, d1, d2 = [int(b) for b in outcome]
    syndrome = (s0, s1)
    syndrome_counts[syndrome] += count

    # Check for logical error: majority vote on data qubits
    data_sum = d0 + d1 + d2
    logical = 1 if data_sum >= 2 else 0  # Majority vote

    # In ideal case, all data should be same, so logical should match d0
    # With errors, we might see logical errors
    total += count

print("Syndrome distribution (with 1% noise):")
for syndrome, count in sorted(syndrome_counts.items()):
    s_bits = "".join(str(s) for s in syndrome)
    print(f"  syndrome={s_bits}: {count:,} ({100*count/100_000:.2f}%)")

print("\nNote: Non-trivial syndromes indicate detected errors!")

## Summary: Noisy Symbolic Execution

The `pecos.experimental` module now also provides noisy execution:

- `execute_hugr_symbolic_noisy(hugr_bytes, p1, p2, p_meas, p_prep)` - Noisy HUGR execution
- `execute_dag_circuit_symbolic_noisy(circuit, p1, p2, p_meas, p_prep)` - Noisy DagCircuit execution
- `NoisySymbolicExecutionResult` - Result with sampling methods:
  - `.sample(n)` - Get n noisy samples
  - `.sample_counts(n)` - Get outcome counts
  - `.num_faults` - Number of possible fault events
  - `.num_measurements` - Total measurements

**Key advantages of noisy symbolic execution:**
1. **Speed**: Orders of magnitude faster than Monte Carlo for large shot counts
2. **Accuracy**: Produces statistically equivalent distributions
3. **Scalability**: Build once, sample millions of times
4. **Insight**: See explicit fault dependencies in the symbolic structure

**Use cases:**
- Error correction threshold studies (need many shots)
- Decoder training data generation
- Statistical analysis of noisy Clifford circuits
- Syndrome pattern analysis